# Notebook Overview — Run Baseline VideoQA

## Purpose

This notebook performs baseline Video Question Answering (VideoQA) inference on the NExT-QA benchmark using the Qwen2-VL-7B multimodal vision-language model. The notebook establishes a baseline against which the representation-based VideoQA pipelines are compared.

A configurable development subset of the NExT-QA validation split is used to support rapid experimentation, debugging, and performance evaluation before executing full-dataset experiments.

The notebook performs the complete baseline inference workflow:

* Prepare the execution environment and restore required datasets.
* Configure baseline inference parameters.
* Verify GPU availability and model dependencies.
* Load the Qwen2-VL-7B model and processor.
* Prepare the development evaluation dataset.
* Execute baseline multiple-choice VideoQA inference.
* Validate generated prediction artifacts.
* Save experiment artifacts locally.
* Promote artifacts to Google Drive for downstream evaluation.
* Display representative prediction examples.
* Prepare experiment outputs for Notebook 08 evaluation.

## Inputs

* NExT-QA annotations
* NExT-QA video dataset
* Qwen2-VL-7B model
* Development experiment configuration

## Outputs

* Baseline prediction artifacts
* Prediction validation report
* Experiment summary
* Google Drive experiment artifacts

## Downstream Consumer

Notebook 08 — Evaluate Development Results


### 🔷 Step 1 — Initialize Environment and Restore Dataset

* Initialize the notebook runtime and prepare the project execution environment.
* Clone the project repository using sparse checkout to minimize download size and startup overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Load project configuration settings, utility modules, and required input/output paths.
* Mount Google Drive and restore the NExT-QA video dataset from the project release archive when needed.
* Verify local video cache availability and confirm the expected number of video files are present.
* Load NExT-QA question annotations and build the local video inventory.
* Validate annotation coverage and dataset readiness before VideoQA inference begins.
* Optionally display configuration details, dataset statistics, and validation summaries when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Initialize Environment and Restore Dataset
# ============================================================

# Configure diagnostic output and GPU requirements.
VERBOSE = True
REQUIRE_L4_GPU = True

# ------------------------------------------------------------
# Import Dependencies
# ------------------------------------------------------------

# Import standard-library utilities.
import os
import shutil
from pathlib import Path

# Import tabular data utilities.
import pandas as pd

# Import Google Colab services.
from google.colab import userdata, drive

print("Initializing Notebook 01 environment...")
print("-" * 60)

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

# Define the Google Drive mount point.
GOOGLE_DRIVE_MOUNT = "/content/drive"

# Mount Google Drive when necessary.
if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# Clone Required Repository Files
# ------------------------------------------------------------

# Define the project repository location.
REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(
    REPO_BASE_DIR,
    REPO_NAME,
)

# Retrieve the GitHub access token.
github_token = userdata.get("GITHUB_TOKEN")

# Verify that the access token is available.
if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

# Build the authenticated repository URL.
repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# Switch to the Colab workspace.
os.chdir(REPO_BASE_DIR)

# Clone the repository when necessary.
if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    # Create a sparse repository checkout.
    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    # Restore the required project directories.
    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    # Reuse the existing repository checkout.
    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# Load Project Configuration and Utilities
# ------------------------------------------------------------

print("\nLoading project configuration...")

# Import shared project configuration.
from src.videoqa_representation_config import *

# Configure the baseline experiment.
EXPERIMENT_NAME = "qwen2vl_baseline_dev100"
configure_experiment(EXPERIMENT_NAME)

# Import dataset restoration and metadata utilities.
from src.nextqa_video_cache import *
from src.nextqa_metadata import *

# Define required project paths.
required_paths = [
    Path("src"),
    QUESTIONS_DIR,
    METADATA_DIR,
    GOOGLE_DRIVE_ROOT,
]

# Identify missing required paths.
missing_paths = [
    path
    for path in required_paths
    if not Path(path).exists()
]

# Report missing paths and stop initialization.
if missing_paths:

    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

# Create the shared output directory.
OUTPUTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Create the baseline experiment directory.
BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# Restore Local NExT-QA Video Cache
# ------------------------------------------------------------

print("\nChecking local NExT-QA video cache...")

# Inspect the local video cache.
existing_video_files = sorted(
    VIDEOS_DIR.rglob("*.mp4")
)

# Reuse the cache when the expected videos are present.
if len(existing_video_files) == EXPECTED_VIDEO_COUNT:

    print("Local video cache already available.")
    print(f"Videos found: {len(existing_video_files):,}")

else:

    # Restore a missing or incomplete cache.
    print("Local video cache missing or incomplete.")
    print(f"Videos found locally: {len(existing_video_files):,}")
    print("Restoring videos from Google Drive...")

    # Define the Google Drive dataset directory.
    drive_dataset_dir = (
        GOOGLE_DRIVE_ROOT /
        "NExT-QA"
    )

    # Define the Google Drive release directory.
    drive_releases_dir = (
        drive_dataset_dir /
        "releases"
    )

    # Define the local archive directory.
    local_archive_dir = (
        DATASET_DIR /
        "archives"
    )

    local_archive_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Define the source archive path.
    drive_combined_archive_path = (
        drive_releases_dir /
        NEXTQA_COMBINED_ARCHIVE_NAME
    )

    # Define the local archive path.
    local_combined_archive_path = (
        local_archive_dir /
        NEXTQA_COMBINED_ARCHIVE_NAME
    )

    # Verify that the source archive exists.
    if not drive_combined_archive_path.exists():
        raise FileNotFoundError(
            "Missing NExT-QA combined archive in Google Drive:\n"
            f"{drive_combined_archive_path}"
        )

    print("Copying dataset archive from Google Drive...")

    # Copy the archive to local storage.
    shutil.copy2(
        drive_combined_archive_path,
        local_combined_archive_path,
    )

    # Verify the local archive copy.
    if not local_combined_archive_path.exists():
        raise FileNotFoundError(
            "Failed to copy NExT-QA archive locally:\n"
            f"{local_combined_archive_path}"
        )

    print("Extracting or verifying video archive...")

    # Restore the local video cache.
    extract_nextqa_video_archive(
        combined_archive_path=local_combined_archive_path,
        local_videos_dir=VIDEOS_DIR,
        force_extract=False,
        verbose=VERBOSE,
    )

    # Verify the restored video inventory.
    restored_video_files = sorted(
        VIDEOS_DIR.rglob("*.mp4")
    )

    # Require the expected video count.
    if len(restored_video_files) != EXPECTED_VIDEO_COUNT:
        raise ValueError(
            "NExT-QA video cache verification failed. "
            f"Expected {EXPECTED_VIDEO_COUNT:,} videos, "
            f"found {len(restored_video_files):,}."
        )

    print("Video cache restored.")
    print(f"Videos found: {len(restored_video_files):,}")

print("Local NExT-QA video cache ready.")

# ------------------------------------------------------------
# Load NExT-QA Metadata and Video Inventory
# ------------------------------------------------------------

print("\nLoading NExT-QA metadata and video inventory...")

# Load split-specific annotations.
split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

# Combine the annotation splits.
annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

# Build the local video inventory.
video_inventory_df = build_nextqa_video_inventory(
    videos_dir=VIDEOS_DIR,
    verbose=VERBOSE,
)

# Attach video inventory data to the annotations.
annotations_with_videos_df = attach_video_inventory_to_annotations(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

# Summarize annotation counts by split.
split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

# Verify annotation-to-video coverage.
coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

# Report the primary dataset objects.
print("\nDataset metadata ready.")
print(f"Annotation records: {len(annotations_df):,}")
print(f"Video inventory   : {len(video_inventory_df):,}")

# Display the split summary when verbose output is enabled.
if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

# Confirm that initialization is complete.
print("\nNotebook 01 initialization complete.")
print("-" * 60)
print("Ready for Qwen2-VL baseline VideoQA inference.")



### 🔷 Step 2 — Configure Baseline Experiment

* Define the baseline inference configuration used throughout the notebook.
* Configure the NExT-QA evaluation split, development-subset size, and randomization settings.
* Specify multiple-choice answer generation and video frame sampling parameters for baseline VideoQA inference.
* Configure Qwen2-VL generation settings, including output length, temperature, and sampling behavior.
* Define notebook execution options, including intermediate artifact saving and verbose logging.
* Display the active baseline configuration for the current experiment run.



In [ ]:
# ============================================================
# Step 2: Configure Baseline Experiment
# ============================================================

print("Loading baseline inference configuration...")

# ------------------------------------------------------------
# Define Baseline Experiment Configuration
# ------------------------------------------------------------

# Configure the development baseline experiment used throughout
# the notebook. These settings control dataset selection, Qwen2-VL
# inference behavior, and notebook execution options.
BASELINE_CONFIG = {

    # Evaluation dataset configuration.
    "evaluation_split": EVALUATION_SPLIT,
    "development_subset_size": DEVELOPMENT_SUBSET_SIZE,
    "random_seed": RANDOM_SEED,

    # Video Question Answering configuration.
    "answer_mode": ANSWER_MODE,
    "max_frames_per_question": MAX_FRAMES_PER_QUESTION,

    # Qwen2-VL generation parameters.
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "do_sample": DO_SAMPLE,

    # Notebook execution options.
    "save_intermediate_results": True,
    "verbose": True,
}

# ------------------------------------------------------------
# Display Active Configuration
# ------------------------------------------------------------

# Display the active experiment settings before loading the model.
# Recording the configuration in the notebook output makes each
# experiment easier to reproduce and compare.
print("\nBaseline Configuration")
print("-" * 60)

for key, value in BASELINE_CONFIG.items():
    print(f"{key:<30}: {value}")



### 🔷 Step 3 — Verify GPU Runtime and Model Dependencies

* Verify that the runtime satisfies the requirements for Qwen2-VL baseline inference.
* Confirm PyTorch installation and CUDA availability.
* Detect and display GPU hardware information and available GPU memory.
* Verify that required software dependencies are available before model loading.
* Report the runtime configuration used for the current experiment.



In [ ]:
# ============================================================
# Step 3: Verify GPU Runtime and Model Dependencies
# ============================================================

# Import utilities used to inspect the Colab runtime and
# dynamically verify the required Python packages.
import sys
import platform
import importlib

print("Verifying GPU runtime and model dependencies...\n")

# ------------------------------------------------------------
# Runtime Information
# ------------------------------------------------------------

# Display the active Python and operating-system environment.
print("Runtime Information")
print("-" * 60)
print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch / CUDA Verification
# ------------------------------------------------------------

# Verify PyTorch, CUDA availability, detected GPU hardware,
# and the memory state before loading the Qwen2-VL model.
try:
    import torch

    print("\nPyTorch Information")
    print("-" * 60)
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():

        print(f"CUDA Version    : {torch.version.cuda}")
        print(f"GPU Count       : {torch.cuda.device_count()}")

        # Report the name and total memory for every available GPU.
        for idx in range(torch.cuda.device_count()):

            gpu_name = torch.cuda.get_device_name(idx)

            gpu_props = torch.cuda.get_device_properties(idx)

            total_memory_gb = (
                gpu_props.total_memory /
                (1024 ** 3)
            )

            print(
                f"GPU {idx}          : "
                f"{gpu_name}"
            )

            print(
                f"GPU {idx} Memory   : "
                f"{total_memory_gb:.1f} GB"
            )

        # Report the current CUDA memory state before model loading.
        allocated_gb = (
            torch.cuda.memory_allocated() /
            (1024 ** 3)
        )

        reserved_gb = (
            torch.cuda.memory_reserved() /
            (1024 ** 3)
        )

        print(
            f"Allocated Memory : "
            f"{allocated_gb:.2f} GB"
        )

        print(
            f"Reserved Memory  : "
            f"{reserved_gb:.2f} GB"
        )

        # Validate the primary GPU against the notebook requirement.
        primary_gpu = torch.cuda.get_device_name(0)

        if REQUIRE_L4_GPU and "L4" not in primary_gpu:
            raise RuntimeError(
                f"Required NVIDIA L4 GPU not available. "
                f"Detected GPU: {primary_gpu}. "
                "Change the Colab runtime to L4 before continuing."
            )

        # Display hardware-specific guidance for supported Colab GPUs.
        if "T4" in primary_gpu:

            print(
                "\nWARNING: NVIDIA T4 GPU detected "
                "(approximately 16 GB VRAM)."
            )

            print(
                "Large multimodal inference workloads "
                "may require reduced frame counts or "
                "memory optimization settings."
            )

        elif "L4" in primary_gpu:

            print(
                "\nNVIDIA L4 GPU detected "
                "(approximately 24 GB VRAM)."
            )

        device = "cuda"

    else:

        # Preserve a CPU fallback value when CUDA is unavailable.
        print("WARNING: No CUDA GPU detected.")
        device = "cpu"

except Exception as e:

    # Record a CPU fallback when PyTorch cannot be imported or
    # the required GPU validation fails.
    print(f"ERROR: Unable to load PyTorch ({e})")
    device = "cpu"

# ------------------------------------------------------------
# Required Packages
# ------------------------------------------------------------

# Define the packages required by the Qwen2-VL baseline workflow.
required_packages = [
    "transformers",
    "accelerate",
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "PIL",
]

print("\nDependency Verification")
print("-" * 60)

dependency_status = []

# Import each package dynamically and record its availability
# and installed version for the final verification summary.
for package_name in required_packages:

    try:
        module = importlib.import_module(package_name)

        version = getattr(module, "__version__", "unknown")

        dependency_status.append(
            {
                "package": package_name,
                "status": "OK",
                "version": version,
            }
        )

        print(f"[OK]   {package_name:<15} {version}")

    except Exception:
        dependency_status.append(
            {
                "package": package_name,
                "status": "MISSING",
                "version": "",
            }
        )

        print(f"[FAIL] {package_name}")

# ------------------------------------------------------------
# Verification Summary
# ------------------------------------------------------------

# Collect packages that failed validation so the final output
# clearly identifies any missing runtime requirements.
missing_packages = [
    item["package"]
    for item in dependency_status
    if item["status"] != "OK"
]

print("\nVerification Summary")
print("-" * 60)

print(f"Execution Device : {device}")

# Report whether the runtime is ready for model loading.
if len(missing_packages) == 0:
    print("All required dependencies are available.")
else:
    print("Missing packages:")
    for pkg in missing_packages:
        print(f"  - {pkg}")



### 🔷 Step 4 — Load Qwen2-VL-7B Model and Processor

* Load the Qwen2-VL-7B multimodal model used for baseline VideoQA inference.
* Load the associated processor for multimodal input preparation.
* Configure model execution on the available GPU accelerator.
* Verify successful model initialization.
* Display model configuration information for the current experiment.



In [ ]:
# ============================================================
# Step 4: Load Qwen2-VL-7B Model and Processor
# ============================================================

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("Loading Qwen2-VL-7B model and processor...")

# ------------------------------------------------------------
# Verify Model Configuration and GPU Availability
# ------------------------------------------------------------

# Confirm that the shared project configuration was loaded and
# that CUDA is available before attempting to load the model.
if "BASELINE_MODEL_NAME" not in globals():
    raise NameError(
        "BASELINE_MODEL_NAME was not found. "
        "Confirm that videoqa_representation_config.py was loaded in Step 1."
    )

MODEL_ID = BASELINE_MODEL_NAME

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for Qwen2-VL-7B inference. "
        "Please switch Colab runtime to GPU."
    )

device = "cuda"

# ------------------------------------------------------------
# Load Processor and Model
# ------------------------------------------------------------

# Load the multimodal processor responsible for preparing text,
# image, and video inputs in the format expected by Qwen2-VL.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# Load the pretrained model using bfloat16 precision and automatic
# device placement to reduce GPU memory requirements.
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Switch the model to inference mode so training-specific behavior,
# such as dropout, is disabled during baseline prediction.
model.eval()

# ------------------------------------------------------------
# Display Model Summary
# ------------------------------------------------------------

print("Qwen2-VL-7B model and processor loaded successfully.")
print(f"Model ID : {MODEL_ID}")
print(f"Device   : {device}")
print(f"Dtype    : {model.dtype}")



### 🔷 Step 5 — Prepare Development Evaluation Dataset

* Load the configured NExT-QA evaluation split.
* Apply the configured development-subset sampling parameters.
* Validate required annotation fields used for baseline inference.
* Prepare the evaluation dataset for multiple-choice VideoQA prediction.
* Display summary statistics describing the evaluation dataset.



In [ ]:
# ============================================================
# Step 5: Prepare Development Evaluation Dataset
# ============================================================

import random
from pathlib import Path

import pandas as pd

print("Preparing development evaluation subset...")

# Load the active evaluation split, subset size, and random seed
# from the baseline experiment configuration defined in Step 2.
evaluation_split = BASELINE_CONFIG["evaluation_split"]
development_subset_size = BASELINE_CONFIG["development_subset_size"]
random_seed = BASELINE_CONFIG["random_seed"]

# ------------------------------------------------------------
# Select Evaluation Split
# ------------------------------------------------------------

# Verify that the source annotation dataset contains the fields
# required for multiple-choice VideoQA evaluation.
required_annotation_columns = [
    "split",
    "video",
    "question",
    "answer",
    "a0",
    "a1",
    "a2",
    "a3",
    "a4",
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        "annotations_df is missing required columns: "
        f"{missing_annotation_columns}"
    )

# Filter the combined annotation dataset to the configured
# evaluation split before selecting the development videos.
eval_split_df = annotations_df[
    annotations_df["split"] == evaluation_split
].copy()

if eval_split_df.empty:
    raise ValueError(
        f"No records found for split: {evaluation_split}"
    )

# Standardize video identifiers before sampling and filtering.
eval_split_df["video"] = eval_split_df["video"].astype(str)

available_eval_videos = sorted(
    eval_split_df["video"].unique()
)

# Limit the requested subset size when the evaluation split
# contains fewer videos than the configured development target.
selected_video_count = min(
    development_subset_size,
    len(available_eval_videos),
)

print(f"Development subset size   : {development_subset_size:,} videos")
print(f"Available split videos    : {len(available_eval_videos):,}")

# Select videos rather than individual QA records so every question
# associated with each sampled video remains in the evaluation subset.
selected_videos = (
    pd.Series(available_eval_videos)
    .sample(
        n=selected_video_count,
        random_state=random_seed,
    )
    .astype(str)
    .tolist()
)

eval_df = (
    eval_split_df[
        eval_split_df["video"].isin(selected_videos)
    ]
    .copy()
    .sort_values(["video"])
    .reset_index(drop=True)
)

if eval_df.empty:
    raise RuntimeError(
        "Video-based development subset selection produced no QA records."
    )

print(f"Selected evaluation videos : {eval_df['video'].nunique():,}")
print(f"Selected QA records        : {len(eval_df):,}")

# ------------------------------------------------------------
# Attach Video File Paths
# ------------------------------------------------------------

def resolve_video_path(video_id):
    """
    Resolve a NExT-QA video id to a local MP4 path.
    """

    # Search the restored NExT-QA directory recursively because
    # source videos may be stored within nested folders.
    matches = sorted(
        VIDEOS_DIR.rglob(
            f"{video_id}.mp4"
        )
    )

    # Return None when no matching video exists so missing files
    # can be detected and reported together after path resolution.
    if len(matches) == 0:
        return None

    return matches[0]


# Resolve the local video file associated with every selected QA record.
eval_df["video_path"] = (
    eval_df["video"]
    .apply(resolve_video_path)
)

missing_video_count = (
    eval_df["video_path"]
    .isna()
    .sum()
)

print(f"Missing video files        : {missing_video_count}")

# Display representative missing records before stopping the workflow.
if missing_video_count > 0:
    display(
        eval_df[
            eval_df["video_path"].isna()
        ].head()
    )

    raise FileNotFoundError(
        "One or more evaluation samples do not have matching video files."
    )

# ------------------------------------------------------------
# Attach Ground-Truth Answer Text
# ------------------------------------------------------------

def answer_index_to_text(row):
    """
    Convert the numeric NExT-QA answer index to answer text.
    """

    # Map the numeric answer label to its corresponding candidate
    # column so evaluation artifacts contain both forms.
    answer_idx = int(row["answer"])
    option_col = f"a{answer_idx}"

    if option_col not in row.index:
        raise ValueError(
            f"Answer option column not found: {option_col}"
        )

    return row[option_col]


eval_df["ground_truth_text"] = (
    eval_df.apply(
        answer_index_to_text,
        axis=1,
    )
)

# ------------------------------------------------------------
# Attach Stable Question Identifiers When Available
# ------------------------------------------------------------

# Preserve an existing question identifier when supplied by the
# annotations; otherwise derive one from another compatible field.
if "question_id" not in eval_df.columns:

    candidate_question_id_columns = [
        "qid",
        "question_id",
        "question_idx",
        "id",
    ]

    available_question_id_columns = [
        column
        for column in candidate_question_id_columns
        if column in eval_df.columns
    ]

    if available_question_id_columns:
        eval_df["question_id"] = (
            eval_df[
                available_question_id_columns[0]
            ]
            .astype(str)
        )
    else:
        # Create deterministic identifiers only when no source
        # question identifier is available in the annotations.
        eval_df["question_id"] = [
            f"{row.video}_{index}"
            for index, row in eval_df.reset_index().iterrows()
        ]

# ------------------------------------------------------------
# Final Validation
# ------------------------------------------------------------

# Confirm that the prepared evaluation dataset contains the
# complete schema required by the baseline inference step.
required_eval_columns = [
    "question_id",
    "video",
    "question",
    "answer",
    "a0",
    "a1",
    "a2",
    "a3",
    "a4",
    "ground_truth_text",
    "video_path",
]

missing_columns = [
    column
    for column in required_eval_columns
    if column not in eval_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required evaluation columns: {missing_columns}"
    )

# Verify that filtering retained exactly the number of sampled videos.
if eval_df["video"].nunique() != selected_video_count:
    raise ValueError(
        "Selected evaluation video count does not match expected count. "
        f"Expected {selected_video_count}, "
        f"found {eval_df['video'].nunique()}."
    )

# ------------------------------------------------------------
# Display Evaluation Dataset Summary
# ------------------------------------------------------------

print("\nEvaluation dataset prepared successfully.")
print(f"Evaluation videos  : {eval_df['video'].nunique():,}")
print(f"Evaluation samples : {len(eval_df):,}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Random seed        : {random_seed}")
print(f"Answer mode        : {BASELINE_CONFIG['answer_mode']}")

print("\nEvaluation Dataset Preview:")
display(eval_df.head())



### 🔷 Step 6 — Run Development-Subset Baseline VideoQA Inference

* Sample representative video frames from each evaluation video.
* Construct multimodal prompts containing video frames, questions, and answer choices.
* Execute multiple-choice VideoQA inference using Qwen2-VL-7B.
* Record predicted answers together with experiment metadata and runtime statistics.
* Generate the baseline prediction artifacts used by downstream evaluation notebooks.



In [ ]:
# ============================================================
# Step 6: Run Development-Subset Baseline VideoQA Inference
# ============================================================

import time
import gc
import re
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image
import cv2
import torch

# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def clear_gpu_memory():
    """
    Release unused Python and CUDA memory between inference samples.
    """

    # Clear Python objects first, then release unused CUDA memory
    # to reduce accumulation across sequential VideoQA samples.
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def sample_video_frames(
    video_path,
    num_frames=8,
):
    """
    Uniformly sample frames from a video.
    """

    # Open the source video and determine the total number of frames
    # available for uniform temporal sampling.
    cap = cv2.VideoCapture(
        str(video_path)
    )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    # Return an empty frame list when the video cannot be read.
    if frame_count <= 0:
        cap.release()
        return []

    # Select evenly spaced frame positions across the complete video.
    frame_indices = [
        int(i * frame_count / num_frames)
        for i in range(num_frames)
    ]

    frames = []

    # Read each selected frame and convert it from OpenCV BGR
    # format to the RGB PIL format expected by the processor.
    for frame_index in frame_indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_index,
        )

        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB,
            )

            frames.append(
                Image.fromarray(frame)
            )

    cap.release()

    return frames


def build_videoqa_prompt(
    row,
    answer_mode,
):
    """
    Build the text prompt for baseline VideoQA.
    """

    question = row["question"]

    # Format the NExT-QA question and its five candidate answers
    # as a constrained multiple-choice instruction.
    if answer_mode == "multiple_choice":

        choice_text = (
            f"0. {row['a0']}\n"
            f"1. {row['a1']}\n"
            f"2. {row['a2']}\n"
            f"3. {row['a3']}\n"
            f"4. {row['a4']}"
        )

        return (
            "Answer the video question by selecting the best answer choice.\n"
            "Respond with only the number of the best answer choice.\n\n"
            f"Question: {question}\n\n"
            f"Choices:\n{choice_text}"
        )

    # Stop when an unsupported answer format is supplied.
    raise ValueError(
        f"Unsupported answer_mode for this project: {answer_mode}"
    )


def extract_predicted_choice(
    prediction_text,
):
    """
    Extract a predicted multiple-choice answer index from model output.
    """

    if prediction_text is None:
        return None

    text = str(prediction_text).strip()

    # Prefer an answer index at the beginning of the response because
    # the prompt explicitly requests a number-only prediction.
    leading_match = re.match(
        r"^\s*([0-4])\b",
        text,
    )

    if leading_match:
        return int(
            leading_match.group(1)
        )

    # Fall back to the first valid answer index found elsewhere
    # when the model includes additional explanatory text.
    any_match = re.search(
        r"\b([0-4])\b",
        text,
    )

    if any_match:
        return int(
            any_match.group(1)
        )

    return None


def run_videoqa_inference(
    video_path,
    row,
):
    """
    Execute baseline Qwen2-VL inference for one question/video pair.
    """

    # Initialize intermediate objects so they can be released
    # consistently after both successful and failed inference.
    frames = None
    messages = None
    text = None
    inputs = None
    generated_ids = None
    generated_ids_trimmed = None
    output_text = None

    try:

        # Sample the configured number of frames from the source video.
        frames = sample_video_frames(
            video_path=video_path,
            num_frames=BASELINE_CONFIG["max_frames_per_question"],
        )

        if len(frames) == 0:
            return "VIDEO_READ_ERROR"

        # Build the constrained multiple-choice prompt for this QA record.
        prompt_text = build_videoqa_prompt(
            row=row,
            answer_mode=BASELINE_CONFIG["answer_mode"],
        )

        # Construct a multimodal chat message containing the sampled
        # video frames followed by the question and answer choices.
        messages = [
            {
                "role": "user",
                "content": (
                    [
                        {
                            "type": "image",
                            "image": frame,
                        }
                        for frame in frames
                    ]
                    +
                    [
                        {
                            "type": "text",
                            "text": prompt_text,
                        }
                    ]
                ),
            }
        ]

        # Apply the Qwen2-VL chat template and convert the multimodal
        # message into tensors for model inference.
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = processor(
            text=[text],
            images=frames,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
        }

        # Configure deterministic generation unless sampling is
        # explicitly enabled in the baseline experiment settings.
        generate_kwargs = {
            "max_new_tokens": BASELINE_CONFIG["max_new_tokens"],
            "do_sample": BASELINE_CONFIG["do_sample"],
        }

        if BASELINE_CONFIG["do_sample"]:
            generate_kwargs["temperature"] = BASELINE_CONFIG["temperature"]

        # Disable gradient tracking because this step performs inference only.
        with torch.no_grad():

            generated_ids = model.generate(
                **inputs,
                **generate_kwargs,
            )

        # Remove the input prompt tokens before decoding the generated answer.
        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(
                inputs["input_ids"],
                generated_ids,
            )
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        return output_text.strip()

    finally:

        # Release all sample-specific objects before processing
        # the next question/video pair.
        del frames
        del messages
        del text
        del inputs
        del generated_ids
        del generated_ids_trimmed
        del output_text

        clear_gpu_memory()


def optional_value(
    row,
    column_name,
):
    """
    Return a row value when the source annotation column exists.
    """

    # Some NExT-QA annotation variants contain optional metadata
    # fields that should be retained when available.
    if column_name in row.index:
        return row[column_name]

    return None


# ------------------------------------------------------------
# Baseline Inference Loop
# ------------------------------------------------------------

# Confirm that the prepared experiment uses the answer format
# supported by the current baseline workflow.
answer_mode = BASELINE_CONFIG["answer_mode"]

if answer_mode != "multiple_choice":
    raise ValueError(
        "This project currently supports only multiple-choice evaluation."
    )

print(
    "Running development-subset baseline inference "
    f"on {len(eval_df):,} samples..."
)
print(f"Answer mode: {answer_mode}")

prediction_records = []

# Clear residual GPU memory before beginning the timed inference run.
clear_gpu_memory()

start_time = time.time()

# Process each prepared QA record independently so that predictions,
# errors, timing, and metadata can be recorded at the sample level.
for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Running Baseline VideoQA",
):

    try:

        prediction = run_videoqa_inference(
            video_path=row["video_path"],
            row=row,
        )

    except Exception as error:

        # Preserve failed samples in the output artifact rather than
        # interrupting the complete development-subset experiment.
        prediction = f"ERROR: {str(error)}"
        clear_gpu_memory()

    # Convert the source answer and model response to comparable
    # multiple-choice indices.
    ground_truth_choice = int(
        row["answer"]
    )

    predicted_choice = extract_predicted_choice(
        prediction
    )

    choice_correct = (
        predicted_choice == ground_truth_choice
        if predicted_choice is not None
        else False
    )

    # Store one standardized prediction record per QA sample.
    prediction_records.append(
        {
            "pipeline": "baseline_videoqa",
            "prediction_source": "qwen2vl_baseline",
            "model_name": BASELINE_MODEL_NAME,
            "split": evaluation_split,
            "question_id": str(row["question_id"]),
            "video": row["video"],
            "video_id": row["video"],
            "question": row["question"],
            "question_type": optional_value(
                row,
                "type",
            ),
            "answer_mode": answer_mode,
            "a0": row["a0"],
            "a1": row["a1"],
            "a2": row["a2"],
            "a3": row["a3"],
            "a4": row["a4"],
            "ground_truth": row["ground_truth_text"],
            "ground_truth_choice": ground_truth_choice,
            "ground_truth_answer": row["ground_truth_text"],
            "prediction": prediction,
            "predicted_choice": predicted_choice,
            "choice_correct": choice_correct,
            "video_path": str(row["video_path"]),
        }
    )

# ------------------------------------------------------------
# Build Prediction Dataset
# ------------------------------------------------------------

# Convert the accumulated records into the standardized baseline
# prediction dataframe used by validation and artifact-saving steps.
elapsed_time = time.time() - start_time

prediction_df = pd.DataFrame(
    prediction_records
)

baseline_predictions_df = prediction_df.copy()

print(f"Evaluation samples : {len(baseline_predictions_df):,}")
print(f"Elapsed time       : {elapsed_time:.1f} seconds")
print(
    "Average/sample     : "
    f"{elapsed_time / len(baseline_predictions_df):.2f} seconds"
)

# ------------------------------------------------------------
# Calculate Baseline Accuracy
# ------------------------------------------------------------

# Count parseable answer selections and correct predictions,
# then calculate accuracy across the full evaluation subset.
valid_choice_count = (
    baseline_predictions_df["predicted_choice"]
    .notna()
    .sum()
)

correct_choice_count = (
    baseline_predictions_df["choice_correct"]
    .sum()
)

choice_accuracy = (
    correct_choice_count
    / len(baseline_predictions_df)
)

print("\nMultiple-Choice Results")
print("-" * 60)
print(f"Valid choice predictions  : {valid_choice_count:,}")
print(f"Correct choice predictions: {correct_choice_count:,}")
print(f"Choice accuracy           : {choice_accuracy:.2%}")

display(
    baseline_predictions_df.head()
)

# ------------------------------------------------------------
# Build Baseline Summary Artifact
# ------------------------------------------------------------

# Record experiment scope, performance, timing, and generation
# settings in a one-row summary dataset for later comparison.
baseline_summary_df = pd.DataFrame(
    [
        {
            "pipeline": "baseline_videoqa",
            "prediction_source": "qwen2vl_baseline",
            "model_name": BASELINE_MODEL_NAME,
            "split": evaluation_split,
            "answer_mode": answer_mode,
            "evaluation_samples": int(len(baseline_predictions_df)),
            "unique_questions": int(
                baseline_predictions_df[
                    ["video_id", "question_id"]
                ]
                .drop_duplicates()
                .shape[0]
            ),
            "unique_videos": int(
                baseline_predictions_df["video_id"].nunique()
            ),
            "valid_choice_predictions": int(valid_choice_count),
            "correct_choice_predictions": int(correct_choice_count),
            "choice_accuracy": float(choice_accuracy),
            "elapsed_time_seconds": float(elapsed_time),
            "average_time_per_sample_seconds": float(
                elapsed_time / len(baseline_predictions_df)
            ),
            "max_frames_per_question": int(
                BASELINE_CONFIG["max_frames_per_question"]
            ),
            "max_new_tokens": int(
                BASELINE_CONFIG["max_new_tokens"]
            ),
            "do_sample": bool(
                BASELINE_CONFIG["do_sample"]
            ),
        }
    ]
)

print("\nBaseline Summary Artifact")
print("-" * 60)
display(
    baseline_summary_df
)



### 🔷 Step 7 — Validate Baseline Artifacts

* Verify that all baseline prediction artifacts were generated successfully.
* Validate required prediction fields and artifact structure.
* Identify missing predictions, processing failures, and invalid records.
* Generate validation results summarizing artifact integrity.
* Confirm readiness for artifact persistence.



In [ ]:
# ============================================================
# Step 7: Validate Baseline Artifacts
# ============================================================

import pandas as pd

print("Validating baseline prediction artifact...")

# ------------------------------------------------------------
# Verify Required Prediction Artifacts
# ------------------------------------------------------------

# Confirm that the prediction and summary datasets were created
# by the baseline inference step before performing validation.
if "baseline_predictions_df" not in globals():
    raise NameError(
        "baseline_predictions_df was not found. Run Step 6 first."
    )

required_prediction_columns = [
    "pipeline",
    "prediction_source",
    "model_name",
    "split",
    "question_id",
    "video",
    "video_id",
    "question",
    "answer_mode",
    "a0",
    "a1",
    "a2",
    "a3",
    "a4",
    "ground_truth",
    "ground_truth_choice",
    "ground_truth_answer",
    "prediction",
    "predicted_choice",
    "choice_correct",
]

# Verify that the prediction artifact contains the standardized
# fields required by later saving and evaluation steps.
missing_columns = [
    column
    for column in required_prediction_columns
    if column not in baseline_predictions_df.columns
]

if missing_columns:
    raise ValueError(
        "baseline_predictions_df is missing required columns: "
        f"{missing_columns}"
    )

if baseline_predictions_df.empty:
    raise ValueError(
        "baseline_predictions_df is empty."
    )

if "baseline_summary_df" not in globals():
    raise NameError(
        "baseline_summary_df was not found. Run Step 6 first."
    )

if baseline_summary_df.empty:
    raise ValueError(
        "baseline_summary_df is empty."
    )

# ------------------------------------------------------------
# Identify Problem Predictions
# ------------------------------------------------------------

# Mark predictions that are missing, empty, contain an inference
# error, indicate a video-read failure, or could not be parsed.
problem_mask = (
    baseline_predictions_df["prediction"].isna()
    | baseline_predictions_df["prediction"].astype(str).str.strip().eq("")
    | baseline_predictions_df["prediction"].astype(str).str.startswith("ERROR")
    | baseline_predictions_df["prediction"].astype(str).eq("VIDEO_READ_ERROR")
    | baseline_predictions_df["predicted_choice"].isna()
)

problem_prediction_count = int(
    problem_mask.sum()
)

# ------------------------------------------------------------
# Build Validation Artifact
# ------------------------------------------------------------

# Record dataset coverage, prediction integrity, and baseline
# multiple-choice performance as standardized validation checks.
baseline_validation_records = [
    {
        "check_name": "prediction_records",
        "check_value": int(len(baseline_predictions_df)),
    },
    {
        "check_name": "unique_questions",
        "check_value": int(
            baseline_predictions_df[
                ["video_id", "question_id"]
            ]
            .drop_duplicates()
            .shape[0]
        ),
    },
    {
        "check_name": "unique_videos",
        "check_value": int(
            baseline_predictions_df["video_id"].nunique()
        ),
    },
    {
        "check_name": "missing_predictions",
        "check_value": int(
            baseline_predictions_df["prediction"].isna().sum()
        ),
    },
    {
        "check_name": "empty_predictions",
        "check_value": int(
            baseline_predictions_df["prediction"]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),
    },
    {
        "check_name": "error_predictions",
        "check_value": int(
            baseline_predictions_df["prediction"]
            .astype(str)
            .str.startswith("ERROR")
            .sum()
        ),
    },
    {
        "check_name": "video_read_errors",
        "check_value": int(
            baseline_predictions_df["prediction"]
            .astype(str)
            .eq("VIDEO_READ_ERROR")
            .sum()
        ),
    },
    {
        "check_name": "valid_choice_predictions",
        "check_value": int(
            baseline_predictions_df["predicted_choice"]
            .notna()
            .sum()
        ),
    },
    {
        "check_name": "invalid_choice_predictions",
        "check_value": int(
            baseline_predictions_df["predicted_choice"]
            .isna()
            .sum()
        ),
    },
    {
        "check_name": "correct_choice_predictions",
        "check_value": int(
            baseline_predictions_df["choice_correct"]
            .sum()
        ),
    },
    {
        "check_name": "choice_accuracy",
        "check_value": float(
            baseline_predictions_df["choice_correct"]
            .mean()
        ),
    },
    {
        "check_name": "problem_predictions",
        "check_value": problem_prediction_count,
    },
]

baseline_validation_df = pd.DataFrame(
    baseline_validation_records
)

# ------------------------------------------------------------
# Display Validation Results
# ------------------------------------------------------------

print("Baseline prediction validation complete.")
print("-" * 60)

display(
    baseline_validation_df
)

# Display the affected prediction records only when validation
# identifies a missing, invalid, or failed inference result.
if problem_prediction_count > 0:
    print("\nProblem predictions detected:")
    display(
        baseline_predictions_df[
            problem_mask
        ]
    )
else:
    print(
        "\nNo missing, empty, error, video-read-error, "
        "or invalid-choice predictions detected."
    )

# ------------------------------------------------------------
# Runtime Projection
# ------------------------------------------------------------

# Use the observed development-subset inference rate to estimate
# the runtime required to process the complete annotation dataset.
avg_time_per_sample = (
    elapsed_time
    / len(baseline_predictions_df)
)

total_dataset_size = len(
    annotations_df
)

projected_seconds = (
    avg_time_per_sample
    * total_dataset_size
)

projected_hours = (
    projected_seconds
    / 3600
)

print("\nRuntime Projection")
print("-" * 60)
print(f"Average Time per Sample : {avg_time_per_sample:.2f} sec")
print(f"Dataset Size            : {total_dataset_size:,}")
print(f"Projected Runtime       : {projected_hours:.2f} hours")



### 🔷 Step 8 — Save Baseline Artifacts

* Verify that all required baseline artifacts are available for saving.
* Create the local output directory when necessary.
* Save prediction, validation, and summary artifacts to the local experiment directory.
* Report the locations of the generated baseline artifacts.


In [ ]:
# ============================================================
# Step 8: Save Baseline Artifacts
# ============================================================

print("Saving baseline artifacts locally...")

# Verify that the prediction and validation datasets were created
# before attempting to write the experiment artifacts.
if "baseline_predictions_df" not in globals():
    raise NameError(
        "baseline_predictions_df was not found. Run Step 6 first."
    )

if "baseline_validation_df" not in globals():
    raise NameError(
        "baseline_validation_df was not found. Run Step 7 first."
    )

# ------------------------------------------------------------
# Create Local Output Directory
# ------------------------------------------------------------

# Create the configured baseline output directory when it does
# not already exist.
BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Save Baseline CSV Artifacts
# ------------------------------------------------------------

# Save the sample-level predictions, validation checks, and
# experiment summary as separate CSV files.
baseline_predictions_df.to_csv(
    BASELINE_PREDICTIONS_CSV,
    index=False,
)

baseline_validation_df.to_csv(
    BASELINE_VALIDATION_CSV,
    index=False,
)

baseline_summary_df.to_csv(
    BASELINE_SUMMARY_CSV,
    index=False,
)

# ------------------------------------------------------------
# Verify Saved Files
# ------------------------------------------------------------

# Collect the expected local artifacts so their existence can
# be checked before the workflow continues.
local_artifacts = [
    BASELINE_PREDICTIONS_CSV,
    BASELINE_VALIDATION_CSV,
    BASELINE_SUMMARY_CSV,
]

missing_local_artifacts = [
    path
    for path in local_artifacts
    if not path.exists()
]

if missing_local_artifacts:
    raise FileNotFoundError(
        "One or more expected local baseline artifacts were not created: "
        f"{missing_local_artifacts}"
    )

# Display the verified artifact paths for later reference.
print("Local baseline artifacts saved.")
print("-" * 60)

for artifact_path in local_artifacts:
    print(artifact_path)



### 🔷 Step 9 — Promote Baseline Artifacts to Google Drive

* Verify that the required baseline artifacts exist locally.
* Create the Google Drive output directory when necessary.
* Copy prediction, validation, and summary artifacts to Google Drive.
* Confirm successful artifact promotion for downstream analysis.



In [ ]:
# ============================================================
# Step 9: Promote Baseline Artifacts to Google Drive
# ============================================================

import shutil

print("Promoting baseline artifacts to Google Drive...")

# ------------------------------------------------------------
# Define Local and Google Drive Artifact Lists
# ------------------------------------------------------------

# List the locally generated baseline artifacts that will be
# promoted to persistent Google Drive storage.
local_artifacts = [
    BASELINE_PREDICTIONS_CSV,
    BASELINE_VALIDATION_CSV,
    BASELINE_SUMMARY_CSV,
]

# Define the corresponding Google Drive destination paths.
drive_artifacts = [
    BASELINE_PREDICTIONS_DRIVE_CSV,
    BASELINE_VALIDATION_DRIVE_CSV,
    BASELINE_SUMMARY_DRIVE_CSV,
]

# ------------------------------------------------------------
# Verify Local Artifacts Exist
# ------------------------------------------------------------

# Identify any required local artifacts that are missing.
missing_local_artifacts = [
    path
    for path in local_artifacts
    if not path.exists()
]

# Stop execution if one or more required artifacts were not created.
if missing_local_artifacts:
    raise FileNotFoundError(
        "Cannot promote missing local baseline artifacts: "
        f"{missing_local_artifacts}"
    )

# ------------------------------------------------------------
# Create Google Drive Output Directory
# ------------------------------------------------------------

# Ensure the destination directory exists before copying files.
BASELINE_VIDEOQA_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Copy Artifacts to Google Drive
# ------------------------------------------------------------

# Copy each local artifact to its corresponding Google Drive location.
for local_path, drive_path in zip(
    local_artifacts,
    drive_artifacts,
):
    shutil.copy2(
        local_path,
        drive_path,
    )

# ------------------------------------------------------------
# Verify Promoted Artifacts
# ------------------------------------------------------------

# Confirm that every artifact now exists in Google Drive.
missing_drive_artifacts = [
    path
    for path in drive_artifacts
    if not path.exists()
]

# Stop execution if any artifact failed to copy successfully.
if missing_drive_artifacts:
    raise FileNotFoundError(
        "One or more baseline artifacts were not promoted to Drive: "
        f"{missing_drive_artifacts}"
    )

# ------------------------------------------------------------
# Display Promotion Results
# ------------------------------------------------------------

# Report successful promotion and list the Drive artifact locations.
print("Baseline artifacts promoted to Google Drive.")
print("-" * 60)
for artifact_path in drive_artifacts:
    print(artifact_path)



### 🔷 Step 10 — Display Baseline Results

* Display summary statistics describing the completed baseline VideoQA experiment.
* Present representative prediction examples together with the corresponding questions, answer choices, ground-truth answers, and model predictions.
* Review experiment accuracy, inference timing, and generated artifacts.
* Provide a qualitative and quantitative sanity check before downstream evaluation.



In [ ]:
# ============================================================
# Step 10: Display Baseline Results
# ============================================================

print("Displaying baseline results...")

# ------------------------------------------------------------
# Verify Required Result DataFrames
# ------------------------------------------------------------

# Confirm that baseline predictions were generated previously.
if "baseline_predictions_df" not in globals():
    raise NameError(
        "baseline_predictions_df was not found. Run Step 6 first."
    )

# Confirm that baseline validation results are available.
if "baseline_validation_df" not in globals():
    raise NameError(
        "baseline_validation_df was not found. Run Step 7 first."
    )

# Confirm that the baseline summary artifact is available.
if "baseline_summary_df" not in globals():
    raise NameError(
        "baseline_summary_df was not found. Run Step 9 first."
    )

# ------------------------------------------------------------
# Display Summary and Validation Artifacts
# ------------------------------------------------------------

# Display the experiment-level baseline summary.
print("\nBaseline Summary Artifact")
print("-" * 60)
display(
    baseline_summary_df
)

# Display the artifact-validation results.
print("\nBaseline Validation Artifact")
print("-" * 60)
display(
    baseline_validation_df
)

# ------------------------------------------------------------
# Select Representative Prediction Records
# ------------------------------------------------------------

# Limit the displayed prediction sample to at most ten records.
sample_count = min(
    10,
    len(baseline_predictions_df),
)

# Select a reproducible random sample using the configured seed.
sample_predictions_df = (
    baseline_predictions_df
    .sample(
        n=sample_count,
        random_state=BASELINE_CONFIG["random_seed"],
    )
    .reset_index(drop=True)
)

# Define the prediction fields included in the qualitative review.
display_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "ground_truth_choice",
    "predicted_choice",
    "choice_correct",
]

# Display the sampled prediction records.
print(f"\nDisplaying {sample_count} sample predictions...")
print("-" * 60)

display(
    sample_predictions_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Calculate Baseline Choice Accuracy
# ------------------------------------------------------------

# Count the number of correctly predicted answer choices.
correct_choice_predictions = int(
    baseline_predictions_df["choice_correct"]
    .sum()
)

# Calculate accuracy across all baseline prediction records.
choice_accuracy = (
    correct_choice_predictions
    / len(baseline_predictions_df)
)



### 🔷 Step 11 — Finalize Baseline Artifacts for Evaluation

* Verify that the required baseline artifacts are available in Google Drive.
* Organize experiment outputs using the standard project directory structure.
* Prepare prediction, validation, and summary artifacts for Notebook 08 evaluation.
* Display the final artifact locations and completion status.



In [ ]:
# ============================================================
# Step 11: Finalize Baseline Artifacts for Evaluation
# ============================================================

import shutil

# ------------------------------------------------------------
# Verify Required Google Drive Paths
# ------------------------------------------------------------

# Collect the required Google Drive destination paths used for
# final artifact promotion.
required_drive_paths = [
    BASELINE_VIDEOQA_DRIVE_DIR,
    BASELINE_PREDICTIONS_DRIVE_CSV,
    BASELINE_VALIDATION_DRIVE_CSV,
    BASELINE_SUMMARY_DRIVE_CSV,
]

# Verify that all required Drive path constants have been defined.
for drive_path in required_drive_paths:
    if drive_path is None:
        raise ValueError(
            "One or more baseline Drive path constants are undefined."
        )

# ------------------------------------------------------------
# Create Google Drive Output Directory
# ------------------------------------------------------------

# Ensure the destination directory exists before copying artifacts.
BASELINE_VIDEOQA_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Define Artifact Promotion Mapping
# ------------------------------------------------------------

# Pair each locally generated artifact with its corresponding
# Google Drive destination.
promotion_pairs = [
    (
        BASELINE_PREDICTIONS_CSV,
        BASELINE_PREDICTIONS_DRIVE_CSV,
    ),
    (
        BASELINE_VALIDATION_CSV,
        BASELINE_VALIDATION_DRIVE_CSV,
    ),
    (
        BASELINE_SUMMARY_CSV,
        BASELINE_SUMMARY_DRIVE_CSV,
    ),
]

# ------------------------------------------------------------
# Promote Artifacts to Google Drive
# ------------------------------------------------------------

# Verify that each local artifact exists before copying it.
for local_path, drive_path in promotion_pairs:

    if not local_path.exists():
        raise FileNotFoundError(
            f"Cannot promote missing local artifact: {local_path}"
        )

    shutil.copy2(
        local_path,
        drive_path,
    )

# ------------------------------------------------------------
# Verify Promoted Artifacts
# ------------------------------------------------------------

# Confirm that every promoted artifact now exists in Google Drive.
for _, drive_path in promotion_pairs:

    if not drive_path.exists():
        raise FileNotFoundError(
            f"Expected Drive artifact was not created: {drive_path}"
        )

# ------------------------------------------------------------
# Display Promoted Artifact Locations
# ------------------------------------------------------------

# Display the final Google Drive locations for the promoted artifacts.
for _, drive_path in promotion_pairs:
    print(drive_path)

# ------------------------------------------------------------
# Display Notebook Summary
# ------------------------------------------------------------

# Display a summary of the baseline experiment.
print("\nNotebook 01 complete.")
print("=" * 60)

print("\nBaseline VideoQA")
print("-" * 60)
print(f"Model                : {BASELINE_MODEL_NAME}")
print(f"Prediction records   : {len(baseline_predictions_df):,}")
print(
    "Questions evaluated  : "
    f"{baseline_predictions_df[['video_id', 'question_id']].drop_duplicates().shape[0]:,}"
)
print(f"Videos evaluated     : {baseline_predictions_df['video_id'].nunique():,}")
print(f"Correct predictions  : {baseline_predictions_df['choice_correct'].sum():,}")
print(f"Accuracy             : {baseline_predictions_df['choice_correct'].mean():.2%}")

# ------------------------------------------------------------
# Display Notebook 08 Compatibility Information
# ------------------------------------------------------------

# Confirm that the generated artifacts are ready for evaluation
# by Notebook 08.
print("\nNotebook 08 Compatibility")
print("-" * 60)
print(f"Experiment name : {EXPERIMENT_NAME}")
print("Notebook 08 can evaluate these results.")

